# Activation Functions in Neural Networks: An Interactive Guide

## Learning Objectives

By the end of this notebook, you will:
- Understand what activation functions are and **why** they're necessary
- Know the mathematical definition, behavior, and derivatives of 10 common activation functions
- Develop strong intuitions about the pros/cons of each function
- Know when to use each activation function and when to avoid them
- Be able to choose the right activation function for your neural network

## What We'll Cover

1. **Introduction**: Why activation functions matter
2. **10 Activation Functions**: From classic to modern
3. **Comparative Analysis**: Side-by-side comparisons
4. **Practical Demonstrations**: Real neural network training
5. **Decision Guide**: How to choose the right function

Let's begin!

## Part 1: Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from aiml_notebooks import get_device, set_seed

# Set random seed for reproducibility
set_seed(42)

# Get device
device = get_device()

# Plot styling
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

## Part 2: What is an Activation Function?

An **activation function** is a mathematical function applied to the output of each neuron in a neural network. It determines whether and how strongly a neuron should "fire" (activate).

### The Mathematical Form

Given an input $x$, an activation function $f$ produces an output:

$$y = f(x)$$

In a neural network layer:

$$\mathbf{y} = f(\mathbf{W}\mathbf{x} + \mathbf{b})$$

where $\mathbf{W}$ is the weight matrix, $\mathbf{x}$ is the input, and $\mathbf{b}$ is the bias.

### Why Do We Need Activation Functions?

**Critical insight**: Without activation functions (or with only linear activations), a neural network with any number of layers is equivalent to a single-layer linear model!

Let's see why...

In [ ]:
# Demonstration: Stacking linear transformations

# Create a simple 3-layer "network" with only linear transformations
W1 = torch.tensor([[2.0, 1.0], [1.0, 3.0]])
W2 = torch.tensor([[1.5, 0.5], [0.5, 2.0]])
W3 = torch.tensor([[1.0, 2.0], [2.0, 1.0]])

# Sample input
x = torch.tensor([[1.0], [2.0]])

# Forward pass through 3 layers (no activation)
h1 = W1 @ x
h2 = W2 @ h1
y_multilayer = W3 @ h2

# This is equivalent to a single matrix multiplication!
W_combined = W3 @ W2 @ W1
y_singlelayer = W_combined @ x

print("3-layer output:", y_multilayer.squeeze().numpy())
print("Single-layer output:", y_singlelayer.squeeze().numpy())
print("Difference:", (y_multilayer - y_singlelayer).abs().max().item())
print("\n✓ They're identical! Without nonlinearity, depth doesn't help.")

### 🤔 Reflection Question

Why does this matter? Because many real-world problems require **nonlinear decision boundaries**. Linear models can only separate data with straight lines (or hyperplanes). Activation functions give neural networks the power to learn complex, nonlinear patterns.

Let's visualize this...

In [ ]:
# Visualization: Linear vs Nonlinear Decision Boundaries

# Create XOR problem (not linearly separable)
np.random.seed(42)
n_points = 100

# Class 0: top-left and bottom-right
X0_tl = np.random.randn(n_points//4, 2) * 0.3 + np.array([0.5, 1.5])
X0_br = np.random.randn(n_points//4, 2) * 0.3 + np.array([1.5, 0.5])
X0 = np.vstack([X0_tl, X0_br])

# Class 1: top-right and bottom-left
X1_tr = np.random.randn(n_points//4, 2) * 0.3 + np.array([1.5, 1.5])
X1_bl = np.random.randn(n_points//4, 2) * 0.3 + np.array([0.5, 0.5])
X1 = np.vstack([X1_tr, X1_bl])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: The problem
axes[0].scatter(X0[:, 0], X0[:, 1], c='blue', marker='o', s=50, alpha=0.6, label='Class 0')
axes[0].scatter(X1[:, 0], X1[:, 1], c='red', marker='s', s=50, alpha=0.6, label='Class 1')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].set_title('XOR Problem (Non-linearly Separable)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Why linear fails
axes[1].scatter(X0[:, 0], X0[:, 1], c='blue', marker='o', s=50, alpha=0.6, label='Class 0')
axes[1].scatter(X1[:, 0], X1[:, 1], c='red', marker='s', s=50, alpha=0.6, label='Class 1')

# Try to draw linear decision boundaries (they fail)
x_line = np.linspace(0, 2, 100)
axes[1].plot(x_line, x_line, 'g--', linewidth=2, alpha=0.5, label='Linear boundary (fails)')
axes[1].plot(x_line, 2-x_line, 'm--', linewidth=2, alpha=0.5)
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].set_title('Linear Boundaries Cannot Solve This')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Key Insight: Nonlinear activation functions allow neural networks to")
print("   learn complex decision boundaries that linear models cannot.")

## Part 3: Key Properties of Activation Functions

When evaluating activation functions, we consider:

1. **Range**: What values can $f(x)$ produce?
2. **Monotonicity**: Does $f(x)$ always increase (or decrease)?
3. **Smoothness**: Is $f(x)$ differentiable everywhere?
4. **Saturation**: Does the gradient $f'(x) \to 0$ for large $|x|$?
5. **Zero-centered**: Is the output centered around 0?
6. **Computational cost**: How expensive is $f(x)$ to compute?
7. **Gradient behavior**: How well do gradients flow during backpropagation?

These properties directly impact:
- Training speed
- Gradient flow (vanishing/exploding gradients)
- Model expressiveness
- Convergence quality

## Part 4: Helper Functions for Visualization

Let's create utility functions to visualize activation functions consistently.

In [ ]:
def plot_activation_and_derivative(x, y, dy_dx, title, xlim=(-5, 5)):
    """
    Plot an activation function and its derivative side-by-side.
    
    Args:
        x: Input values
        y: Activation function output
        dy_dx: Derivative of activation function
        title: Name of activation function
        xlim: X-axis limits for zooming
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot activation function
    axes[0].plot(x, y, 'b-', linewidth=2.5, label=f'{title}')
    axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
    axes[0].axvline(x=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlabel('Input (x)', fontsize=12)
    axes[0].set_ylabel('Output f(x)', fontsize=12)
    axes[0].set_title(f'{title} Function', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].set_xlim(xlim)
    
    # Plot derivative
    axes[1].plot(x, dy_dx, 'r-', linewidth=2.5, label=f"{title}'")
    axes[1].axhline(y=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
    axes[1].axvline(x=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlabel('Input (x)', fontsize=12)
    axes[1].set_ylabel("Gradient f'(x)", fontsize=12)
    axes[1].set_title(f'{title} Derivative', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=11)
    axes[1].set_xlim(xlim)
    
    plt.tight_layout()
    plt.show()

def print_properties(name, range_str, monotonic, smooth, saturates, zero_centered, cost):
    """
    Print key properties of an activation function in a formatted table.
    """
    print(f"\n{'='*60}")
    print(f"Properties of {name}")
    print(f"{'='*60}")
    print(f"  Range:          {range_str}")
    print(f"  Monotonic:      {monotonic}")
    print(f"  Smooth:         {smooth}")
    print(f"  Saturates:      {saturates}")
    print(f"  Zero-centered:  {zero_centered}")
    print(f"  Cost:           {cost}")
    print(f"{'='*60}\n")

## Part 5: The Activation Functions

Now we'll explore 10 activation functions in detail. For each, we'll cover:
- Mathematical definition
- Derivative
- Visualization
- Properties
- Pros and cons
- When to use

Let's create our input range for all visualizations:

In [ ]:
# Input range for visualizations
x_np = np.linspace(-5, 5, 1000)
x_torch = torch.from_numpy(x_np).float().requires_grad_(True)

---

### 1. Linear (Identity) Activation

**Mathematical Definition:**
$$f(x) = x$$

**Derivative:**
$$f'(x) = 1$$

The simplest activation function. It does nothing to the input!

In [ ]:
# Linear activation
y_linear = x_np
dy_linear = np.ones_like(x_np)

plot_activation_and_derivative(x_np, y_linear, dy_linear, 'Linear')

print_properties(
    name="Linear",
    range_str="(-∞, +∞)",
    monotonic="Yes (always increasing)",
    smooth="Yes",
    saturates="No",
    zero_centered="Yes",
    cost="Very low (no computation)"
)

**✅ Pros:**
- Extremely simple and fast
- No vanishing gradient problem
- Good for output layers in regression tasks

**❌ Cons:**
- **No nonlinearity!** Multiple layers collapse to single layer
- Cannot learn complex patterns
- Makes deep networks useless

**📌 When to Use:**
- **Output layer** for regression problems (predicting continuous values)
- **Never** in hidden layers (defeats the purpose of deep learning)

**💡 Key Insight:** This is our baseline. It shows us what we're missing without nonlinearity!

---

### 2. Sigmoid (Logistic) Activation

**Mathematical Definition:**
$$f(x) = \sigma(x) = \frac{1}{1 + e^{-x}}$$

**Derivative:**
$$f'(x) = f(x)(1 - f(x)) = \sigma(x)(1 - \sigma(x))$$

The classic "S-shaped" activation function. Squashes any input to the range (0, 1).

In [ ]:
# Sigmoid activation
y_sigmoid_torch = torch.sigmoid(x_torch)
y_sigmoid = y_sigmoid_torch.detach().numpy()

# Compute derivative analytically: sigmoid'(x) = sigmoid(x) * (1 - sigmoid(x))
dy_sigmoid = y_sigmoid * (1 - y_sigmoid)

plot_activation_and_derivative(x_np, y_sigmoid, dy_sigmoid, 'Sigmoid')

print_properties(
    name="Sigmoid",
    range_str="(0, 1)",
    monotonic="Yes (always increasing)",
    smooth="Yes (infinitely differentiable)",
    saturates="Yes (both ends)",
    zero_centered="No (outputs always positive)",
    cost="Medium (exponential)"
)

**✅ Pros:**
- Output is bounded (0, 1) - interpretable as probability
- Smooth and differentiable everywhere
- Historically important (used in early neural networks)
- Good for binary classification output layer

**❌ Cons:**
- **Vanishing gradient problem**: Gradients near 0 for large $|x|$ (see derivative plot!)
- **Not zero-centered**: Outputs always positive, slows down training
- **Expensive**: Requires computing exponential
- Saturates on both ends (gradient → 0)

**📌 When to Use:**
- **Output layer** for binary classification
- When you need outputs in (0, 1) range
- **Avoid** in deep hidden layers (use ReLU instead)

**💡 Key Insight:** Notice how the gradient (derivative) approaches 0 at the extremes. This is the **vanishing gradient problem** - gradients get smaller and smaller as they backpropagate through layers, making deep networks hard to train!

---

### 3. Tanh (Hyperbolic Tangent) Activation

**Mathematical Definition:**
$$f(x) = \tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}} = \frac{e^{2x} - 1}{e^{2x} + 1}$$

**Derivative:**
$$f'(x) = 1 - \tanh^2(x)$$

A zero-centered version of sigmoid. Squashes inputs to (-1, 1).

In [ ]:
# Tanh activation
y_tanh_torch = torch.tanh(x_torch)
y_tanh_torch.sum().backward()

y_tanh = y_tanh_torch.detach().numpy()
dy_tanh = x_torch.grad.numpy()
x_torch.grad.zero_()

plot_activation_and_derivative(x_np, y_tanh, dy_tanh, 'Tanh')

print_properties(
    name="Tanh",
    range_str="(-1, 1)",
    monotonic="Yes (always increasing)",
    smooth="Yes (infinitely differentiable)",
    saturates="Yes (both ends)",
    zero_centered="Yes ✓",
    cost="Medium (exponential)"
)

**✅ Pros:**
- **Zero-centered** (outputs can be negative) - better than sigmoid
- Stronger gradients than sigmoid (derivative peaks at 1 vs 0.25)
- Smooth and differentiable
- Often works better than sigmoid in hidden layers

**❌ Cons:**
- Still suffers from **vanishing gradient** (though less than sigmoid)
- Still saturates at both ends
- Computationally expensive (exponentials)

**📌 When to Use:**
- **Hidden layers** in RNNs, LSTMs (very common!)
- When you need outputs in (-1, 1) range
- Prefer over sigmoid for hidden layers (but ReLU is often better)

**💡 Key Insight:** Tanh is basically a scaled and shifted sigmoid. It's strictly better than sigmoid for hidden layers because of zero-centering, but still has the saturation problem.

Let's compare Sigmoid and Tanh side-by-side:

In [ ]:
# Comparison: Sigmoid vs Tanh
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Functions
axes[0].plot(x_np, y_sigmoid, 'b-', linewidth=2.5, label='Sigmoid')
axes[0].plot(x_np, y_tanh, 'r-', linewidth=2.5, label='Tanh')
axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
axes[0].axvline(x=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlabel('Input (x)')
axes[0].set_ylabel('Output')
axes[0].set_title('Sigmoid vs Tanh: Functions')
axes[0].legend()

# Derivatives
axes[1].plot(x_np, dy_sigmoid, 'b-', linewidth=2.5, label="Sigmoid' (max=0.25)")
axes[1].plot(x_np, dy_tanh, 'r-', linewidth=2.5, label="Tanh' (max=1.0)")
axes[1].axhline(y=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
axes[1].axvline(x=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
axes[1].grid(True, alpha=0.3)
axes[1].set_xlabel('Input (x)')
axes[1].set_ylabel('Gradient')
axes[1].set_title('Sigmoid vs Tanh: Derivatives')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n💡 Notice: Tanh has 4x larger maximum gradient (1.0 vs 0.25)!")
print("   This means better gradient flow, but both still saturate.")

---

### 4. ReLU (Rectified Linear Unit)

**Mathematical Definition:**
$$f(x) = \max(0, x) = \begin{cases} x & \text{if } x > 0 \\ 0 & \text{if } x \leq 0 \end{cases}$$

**Derivative:**
$$f'(x) = \begin{cases} 1 & \text{if } x > 0 \\ 0 & \text{if } x \leq 0 \end{cases}$$

The **modern standard** for hidden layers. Simple yet powerful!

In [ ]:
# ReLU activation
y_relu_torch = F.relu(x_torch)
y_relu_torch.sum().backward()

y_relu = y_relu_torch.detach().numpy()
dy_relu = x_torch.grad.numpy()
x_torch.grad.zero_()

plot_activation_and_derivative(x_np, y_relu, dy_relu, 'ReLU')

print_properties(
    name="ReLU",
    range_str="[0, +∞)",
    monotonic="Yes (non-decreasing)",
    smooth="No (not differentiable at x=0)",
    saturates="Only on left (x < 0)",
    zero_centered="No",
    cost="Very low (just max operation)"
)

**✅ Pros:**
- **No vanishing gradient** for positive inputs (gradient = 1)
- **Extremely fast** to compute (just `max(0, x)`)
- **Sparse activation**: About 50% of neurons are zero (efficient!)
- Works very well in practice (most popular activation)
- Doesn't saturate for positive values

**❌ Cons:**
- **Dying ReLU problem**: Neurons can "die" (always output 0)
- Not zero-centered (all outputs ≥ 0)
- Not differentiable at x=0 (though we typically set gradient to 0 or 1)
- Unbounded output (can cause instability)

**📌 When to Use:**
- **Default choice** for hidden layers in most neural networks
- CNNs (convolutional neural networks)
- Deep feedforward networks
- When training speed matters

**💡 Key Insight:** ReLU revolutionized deep learning! Its simplicity and lack of vanishing gradients made training deep networks practical. The "dying ReLU" problem occurs when a neuron's weights get updated such that it always outputs negative values (and thus 0 after ReLU), stopping all learning for that neuron.

### 🔬 Interactive Experiment: Dying ReLU

Let's simulate the dying ReLU problem:

In [ ]:
# Demonstration of dying ReLU
torch.manual_seed(42)

# Create a simple neuron with large negative bias
x_sample = torch.randn(100, 10)  # 100 samples, 10 features
w = torch.randn(10, 1) * 0.1
b = torch.tensor([-5.0])  # Large negative bias

# Forward pass
z = x_sample @ w + b
a = F.relu(z)

# How many activations are zero?
dead_percentage = (a == 0).float().mean().item() * 100

print(f"Percentage of dead (zero) activations: {dead_percentage:.1f}%")
print(f"\nWith a large negative bias ({b.item()}), most inputs become negative")
print("after the linear transformation, and ReLU zeros them out.")
print("\nIf this happens during training, the gradient is 0 and the neuron")
print("stops learning entirely - it's 'dead'!")

# Visualize
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.hist(z.detach().numpy(), bins=30, alpha=0.7, color='blue', edgecolor='black')
plt.axvline(x=0, color='red', linestyle='--', linewidth=2, label='ReLU threshold')
plt.xlabel('Pre-activation (z = Wx + b)')
plt.ylabel('Frequency')
plt.title('Before ReLU: Most values are negative')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(a.detach().numpy(), bins=30, alpha=0.7, color='green', edgecolor='black')
plt.xlabel('Activation (a = ReLU(z))')
plt.ylabel('Frequency')
plt.title(f'After ReLU: {dead_percentage:.1f}% are dead (zero)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

### 5. Leaky ReLU

**Mathematical Definition:**
$$f(x) = \begin{cases} x & \text{if } x > 0 \\ \alpha x & \text{if } x \leq 0 \end{cases}$$

where $\alpha$ is a small constant (typically 0.01).

**Derivative:**
$$f'(x) = \begin{cases} 1 & \text{if } x > 0 \\ \alpha & \text{if } x \leq 0 \end{cases}$$

A fix for the dying ReLU problem - allows small negative values through.

In [ ]:
# Leaky ReLU activation
alpha = 0.01
y_leaky_relu_torch = F.leaky_relu(x_torch, negative_slope=alpha)
y_leaky_relu_torch.sum().backward()

y_leaky_relu = y_leaky_relu_torch.detach().numpy()
dy_leaky_relu = x_torch.grad.numpy()
x_torch.grad.zero_()

plot_activation_and_derivative(x_np, y_leaky_relu, dy_leaky_relu, 'Leaky ReLU')

print_properties(
    name="Leaky ReLU",
    range_str="(-∞, +∞)",
    monotonic="Yes (always increasing)",
    smooth="No (kink at x=0)",
    saturates="No ✓",
    zero_centered="No",
    cost="Very low"
)

print(f"Note: Using α = {alpha} (small negative slope for x < 0)")

**✅ Pros:**
- **Fixes dying ReLU**: Neurons can't die (always have gradient)
- Still very fast to compute
- No saturation
- All benefits of ReLU, minus the dying problem

**❌ Cons:**
- Still not zero-centered
- Requires tuning $\alpha$ hyperparameter (though 0.01 works well)
- Slightly more computation than ReLU (negligible)

**📌 When to Use:**
- When you observe dying ReLU problems
- As a safer default than ReLU
- Same use cases as ReLU (CNNs, deep networks)

**💡 Key Insight:** The small negative slope ($\alpha = 0.01$) allows gradients to flow even for negative inputs, preventing neurons from dying. It's like a safety net!

In [ ]:
# Compare ReLU vs Leaky ReLU
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Focus on negative region
x_neg = x_np[x_np <= 1]
y_relu_neg = y_relu[x_np <= 1]
y_leaky_relu_neg = y_leaky_relu[x_np <= 1]

# Functions
axes[0].plot(x_neg, y_relu_neg, 'b-', linewidth=2.5, label='ReLU (dies at x<0)')
axes[0].plot(x_neg, y_leaky_relu_neg, 'r-', linewidth=2.5, label='Leaky ReLU (survives)')
axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
axes[0].axvline(x=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlabel('Input (x)')
axes[0].set_ylabel('Output')
axes[0].set_title('ReLU vs Leaky ReLU: Negative Region')
axes[0].legend()
axes[0].set_xlim(-5, 1)

# Derivatives
dy_relu_neg = dy_relu[x_np <= 1]
dy_leaky_relu_neg = dy_leaky_relu[x_np <= 1]

axes[1].plot(x_neg, dy_relu_neg, 'b-', linewidth=2.5, label="ReLU' = 0 (no gradient!)")
axes[1].plot(x_neg, dy_leaky_relu_neg, 'r-', linewidth=2.5, label=f"Leaky ReLU' = {alpha}")
axes[1].axhline(y=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
axes[1].axvline(x=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
axes[1].grid(True, alpha=0.3)
axes[1].set_xlabel('Input (x)')
axes[1].set_ylabel('Gradient')
axes[1].set_title('Gradients: Leaky ReLU Never Dies')
axes[1].legend()
axes[1].set_xlim(-5, 1)
axes[1].set_ylim(-0.05, 0.15)

plt.tight_layout()
plt.show()

print("\n💡 The small negative slope in Leaky ReLU ensures gradient flow,")
print("   preventing neurons from dying completely.")

---

### 6. ELU (Exponential Linear Unit)

**Mathematical Definition:**
$$f(x) = \begin{cases} x & \text{if } x > 0 \\ \alpha(e^x - 1) & \text{if } x \leq 0 \end{cases}$$

where $\alpha$ is typically 1.0.

**Derivative:**
$$f'(x) = \begin{cases} 1 & \text{if } x > 0 \\ f(x) + \alpha & \text{if } x \leq 0 \end{cases}$$

A smooth alternative to ReLU that pushes mean activations closer to zero.

In [ ]:
# ELU activation
alpha_elu = 1.0
y_elu_torch = F.elu(x_torch, alpha=alpha_elu)
y_elu_torch.sum().backward()

y_elu = y_elu_torch.detach().numpy()
dy_elu = x_torch.grad.numpy()
x_torch.grad.zero_()

plot_activation_and_derivative(x_np, y_elu, dy_elu, 'ELU')

print_properties(
    name="ELU",
    range_str="(-α, +∞) where α=1.0 → (-1, +∞)",
    monotonic="Yes (always increasing)",
    smooth="Yes ✓ (smooth everywhere)",
    saturates="Soft saturation for x << 0",
    zero_centered="Nearly (negative values possible)",
    cost="Medium (exponential for x<0)"
)

**✅ Pros:**
- **Smooth** (no kink, unlike ReLU)
- **Closer to zero-centered** (can output negative values)
- No dying neurons (like Leaky ReLU)
- Often faster convergence than ReLU
- Robust to noise

**❌ Cons:**
- **More expensive** than ReLU (exponential for negative inputs)
- Slight saturation for very negative values
- Less popular than ReLU (less tested in practice)

**📌 When to Use:**
- When you want smoother gradients than ReLU
- When faster convergence is more important than speed
- As an alternative to Leaky ReLU with better properties

**💡 Key Insight:** ELU's smoothness helps optimization, and its negative saturation at -α pushes mean activations closer to zero, which can speed up learning.

---

### 7. GELU (Gaussian Error Linear Unit)

**Mathematical Definition (Exact):**
$$f(x) = x \cdot \Phi(x)$$

where $\Phi(x)$ is the cumulative distribution function of the standard normal distribution.

**Approximation (used in practice):**
$$f(x) \approx x \cdot \sigma(1.702x)$$

or

$$f(x) \approx 0.5x\left(1 + \tanh\left[\sqrt{2/\pi}(x + 0.044715x^3)\right]\right)$$

Used in modern Transformers (BERT, GPT)!

In [ ]:
# GELU activation
y_gelu_torch = F.gelu(x_torch)
y_gelu_torch.sum().backward()

y_gelu = y_gelu_torch.detach().numpy()
dy_gelu = x_torch.grad.numpy()
x_torch.grad.zero_()

plot_activation_and_derivative(x_np, y_gelu, dy_gelu, 'GELU')

print_properties(
    name="GELU",
    range_str="(-0.17, +∞) approximately",
    monotonic="Yes (always increasing)",
    smooth="Yes ✓ (infinitely differentiable)",
    saturates="Soft saturation for x << 0",
    zero_centered="Nearly",
    cost="Medium-High (involves erf or tanh)"
)

**✅ Pros:**
- **State-of-the-art** performance in Transformers
- Smooth (unlike ReLU)
- Probabilistic interpretation (weights inputs by their value)
- Works extremely well for NLP tasks
- Non-monotonic derivative (unique property)

**❌ Cons:**
- More expensive to compute than ReLU
- Less interpretable than simpler functions
- Requires approximation for efficiency

**📌 When to Use:**
- **Transformers** and NLP models (BERT, GPT, etc.)
- When you want smooth, probabilistic activation
- Modern architectures where it's the standard

**💡 Key Insight:** GELU can be thought of as a smooth ReLU that weights inputs by their magnitude relative to other inputs (via the Gaussian CDF). It's become the default in many Transformer architectures.

---

### 8. Swish / SiLU (Sigmoid Linear Unit)

**Mathematical Definition:**
$$f(x) = x \cdot \sigma(x) = \frac{x}{1 + e^{-x}}$$

**Derivative:**
$$f'(x) = f(x) + \sigma(x)(1 - f(x))$$

Also known as SiLU (Sigmoid Linear Unit). Self-gated activation discovered by neural architecture search.

In [ ]:
# Swish/SiLU activation
y_silu_torch = F.silu(x_torch)  # SiLU = Swish
y_silu_torch.sum().backward()

y_silu = y_silu_torch.detach().numpy()
dy_silu = x_torch.grad.numpy()
x_torch.grad.zero_()

plot_activation_and_derivative(x_np, y_silu, dy_silu, 'Swish/SiLU')

print_properties(
    name="Swish/SiLU",
    range_str="(-0.28, +∞) approximately",
    monotonic="No! (non-monotonic for x < 0)",
    smooth="Yes ✓",
    saturates="Soft saturation for x << 0",
    zero_centered="Nearly",
    cost="Medium (sigmoid computation)"
)

**✅ Pros:**
- **Smooth** and self-gated
- Often outperforms ReLU in deep networks
- Non-monotonic (allows more complex representations)
- Good performance across many tasks
- Used in modern architectures (EfficientNet, etc.)

**❌ Cons:**
- More expensive than ReLU (requires sigmoid)
- Less interpretable (why does self-gating help?)
- Still relatively new (less battle-tested)

**📌 When to Use:**
- Deep CNNs (EfficientNet uses it)
- When you want smooth, self-gating behavior
- As a modern alternative to ReLU

**💡 Key Insight:** Swish is "self-gated" - it multiplies input by its sigmoid, allowing the network to learn when to let information through. The non-monotonicity (notice the slight dip for negative values) is unusual but beneficial.

---

### 9. Softmax Activation

**Mathematical Definition:**
$$f(\mathbf{x})_i = \frac{e^{x_i}}{\sum_{j=1}^{K} e^{x_j}}$$

where $\mathbf{x}$ is a vector of $K$ values.

**Key Properties:**
- Outputs sum to 1.0
- All outputs in (0, 1)
- Converts logits to probability distribution

**Note:** Softmax operates on vectors, not scalars, so we'll visualize it differently.

In [ ]:
# Softmax visualization (different from previous functions)

# Create sample logits for 3 classes
logits = torch.tensor([[2.0, 1.0, 0.1]])  # Shape: (1, 3)
probabilities = F.softmax(logits, dim=1)

print("Example: 3-class classification")
print(f"Input logits:     {logits.squeeze().numpy()}")
print(f"Softmax outputs:  {probabilities.squeeze().numpy()}")
print(f"Sum of outputs:   {probabilities.sum().item():.6f}")

# Visualize how softmax responds to different inputs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Experiment 1: Vary one logit, keep others fixed
x1_range = np.linspace(-3, 3, 100)
softmax_outputs = np.zeros((100, 3))

for i, x1 in enumerate(x1_range):
    logits_temp = torch.tensor([[x1, 0.0, 0.0]])
    probs_temp = F.softmax(logits_temp, dim=1)
    softmax_outputs[i] = probs_temp.squeeze().numpy()

axes[0].plot(x1_range, softmax_outputs[:, 0], 'b-', linewidth=2.5, label='Class 1 (varied)')
axes[0].plot(x1_range, softmax_outputs[:, 1], 'r-', linewidth=2.5, label='Class 2 (fixed at 0)')
axes[0].plot(x1_range, softmax_outputs[:, 2], 'g-', linewidth=2.5, label='Class 3 (fixed at 0)')
axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
axes[0].axvline(x=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
axes[0].set_xlabel('Logit for Class 1', fontsize=12)
axes[0].set_ylabel('Probability', fontsize=12)
axes[0].set_title('Softmax: Varying One Logit', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 1)

# Experiment 2: Temperature effect
logits_fixed = torch.tensor([[2.0, 1.0, 0.1]])
temperatures = [0.5, 1.0, 2.0, 5.0]
colors = ['purple', 'blue', 'green', 'orange']

for temp, color in zip(temperatures, colors):
    probs_temp = F.softmax(logits_fixed / temp, dim=1).squeeze().numpy()
    axes[1].bar(np.arange(3) + temperatures.index(temp) * 0.2, probs_temp, 
                width=0.2, alpha=0.7, label=f'T={temp}', color=color)

axes[1].set_xlabel('Class', fontsize=12)
axes[1].set_ylabel('Probability', fontsize=12)
axes[1].set_title('Softmax Temperature Effect', fontsize=14, fontweight='bold')
axes[1].set_xticks(np.arange(3) + 0.3)
axes[1].set_xticklabels(['Class 0', 'Class 1', 'Class 2'])
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

print("\n💡 Lower temperature → sharper distribution (more confident)")
print("   Higher temperature → softer distribution (less confident)")

**✅ Pros:**
- **Perfect for classification**: Outputs are probabilities
- Differentiable (can backpropagate)
- Emphasizes largest values (competitive)
- Works with any number of classes

**❌ Cons:**
- Only for output layer (not hidden layers)
- Can be sensitive to large logit values (overflow)
- Computationally expensive (multiple exponentials)

**📌 When to Use:**
- **Output layer** for multi-class classification (required!)
- Attention mechanisms in Transformers
- Anywhere you need a probability distribution

**💡 Key Insight:** Softmax is special - it's not applied element-wise but across a vector. It converts arbitrary real values into a valid probability distribution. The temperature parameter controls how "confident" the distribution is.

---

### 10. Softplus Activation

**Mathematical Definition:**
$$f(x) = \ln(1 + e^x)$$

**Derivative:**
$$f'(x) = \frac{e^x}{1 + e^x} = \sigma(x)$$

A smooth approximation of ReLU. Always positive and smooth!

In [ ]:
# Softplus activation
y_softplus_torch = F.softplus(x_torch)
y_softplus_torch.sum().backward()

y_softplus = y_softplus_torch.detach().numpy()
dy_softplus = x_torch.grad.numpy()
x_torch.grad.zero_()

plot_activation_and_derivative(x_np, y_softplus, dy_softplus, 'Softplus')

print_properties(
    name="Softplus",
    range_str="(0, +∞)",
    monotonic="Yes (always increasing)",
    smooth="Yes ✓ (infinitely differentiable)",
    saturates="Slightly for x << 0",
    zero_centered="No",
    cost="Medium (exponential and log)"
)

**✅ Pros:**
- **Smooth approximation of ReLU** (no kink)
- Always positive (good for some applications)
- Derivative is sigmoid (well-behaved)
- No dying neurons

**❌ Cons:**
- More expensive than ReLU (exp and log)
- Less popular (ReLU works well enough)
- Can have vanishing gradient for very negative inputs
- Not zero-centered

**📌 When to Use:**
- When you need smooth, positive activations
- Variational autoencoders (for positive variance parameters)
- As a smooth ReLU alternative

**💡 Key Insight:** Softplus is the smooth version of ReLU - it approaches 0 for negative x and x for positive x, but without the sharp corner at 0.

In [ ]:
# Compare ReLU and Softplus
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Functions
axes[0].plot(x_np, y_relu, 'b-', linewidth=2.5, label='ReLU (sharp corner)')
axes[0].plot(x_np, y_softplus, 'r-', linewidth=2.5, label='Softplus (smooth)')
axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
axes[0].axvline(x=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlabel('Input (x)')
axes[0].set_ylabel('Output')
axes[0].set_title('ReLU vs Softplus: Functions')
axes[0].legend()
axes[0].set_xlim(-3, 3)
axes[0].set_ylim(-0.5, 3)

# Derivatives
axes[1].plot(x_np, dy_relu, 'b-', linewidth=2.5, label="ReLU' (discontinuous)")
axes[1].plot(x_np, dy_softplus, 'r-', linewidth=2.5, label="Softplus' = Sigmoid")
axes[1].axhline(y=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
axes[1].axvline(x=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
axes[1].grid(True, alpha=0.3)
axes[1].set_xlabel('Input (x)')
axes[1].set_ylabel('Gradient')
axes[1].set_title('ReLU vs Softplus: Derivatives')
axes[1].legend()
axes[1].set_xlim(-3, 3)

plt.tight_layout()
plt.show()

print("\n💡 Softplus smooths out the corner of ReLU at x=0.")
print("   For large |x|, they behave almost identically.")

## Part 6: Comparative Analysis

Now let's compare all activation functions side-by-side to build deeper intuition.

### All Functions Together

In [ ]:
# Plot all activation functions together
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# All functions
axes[0].plot(x_np, y_linear, linewidth=2, label='Linear', alpha=0.8)
axes[0].plot(x_np, y_sigmoid, linewidth=2, label='Sigmoid', alpha=0.8)
axes[0].plot(x_np, y_tanh, linewidth=2, label='Tanh', alpha=0.8)
axes[0].plot(x_np, y_relu, linewidth=2, label='ReLU', alpha=0.8)
axes[0].plot(x_np, y_leaky_relu, linewidth=2, label='Leaky ReLU', alpha=0.8)
axes[0].plot(x_np, y_elu, linewidth=2, label='ELU', alpha=0.8)
axes[0].plot(x_np, y_gelu, linewidth=2, label='GELU', alpha=0.8)
axes[0].plot(x_np, y_silu, linewidth=2, label='Swish/SiLU', alpha=0.8)
axes[0].plot(x_np, y_softplus, linewidth=2, label='Softplus', alpha=0.8, linestyle='--')

axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
axes[0].axvline(x=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlabel('Input (x)', fontsize=12)
axes[0].set_ylabel('Output f(x)', fontsize=12)
axes[0].set_title('All Activation Functions Compared', fontsize=14, fontweight='bold')
axes[0].legend(loc='upper left', fontsize=10)
axes[0].set_xlim(-5, 5)
axes[0].set_ylim(-2, 5)

# All derivatives
axes[1].plot(x_np, dy_linear, linewidth=2, label="Linear'", alpha=0.8)
axes[1].plot(x_np, dy_sigmoid, linewidth=2, label="Sigmoid'", alpha=0.8)
axes[1].plot(x_np, dy_tanh, linewidth=2, label="Tanh'", alpha=0.8)
axes[1].plot(x_np, dy_relu, linewidth=2, label="ReLU'", alpha=0.8)
axes[1].plot(x_np, dy_leaky_relu, linewidth=2, label="Leaky ReLU'", alpha=0.8)
axes[1].plot(x_np, dy_elu, linewidth=2, label="ELU'", alpha=0.8)
axes[1].plot(x_np, dy_gelu, linewidth=2, label="GELU'", alpha=0.8)
axes[1].plot(x_np, dy_silu, linewidth=2, label="Swish/SiLU'", alpha=0.8)
axes[1].plot(x_np, dy_softplus, linewidth=2, label="Softplus'", alpha=0.8, linestyle='--')

axes[1].axhline(y=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
axes[1].axvline(x=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
axes[1].grid(True, alpha=0.3)
axes[1].set_xlabel('Input (x)', fontsize=12)
axes[1].set_ylabel("Gradient f'(x)", fontsize=12)
axes[1].set_title('All Derivatives Compared', fontsize=14, fontweight='bold')
axes[1].legend(loc='upper left', fontsize=10)
axes[1].set_xlim(-5, 5)
axes[1].set_ylim(-0.2, 1.5)

plt.tight_layout()
plt.show()

### 🤔 Reflection Questions

Looking at the plots above:

1. **Which functions saturate** (gradient → 0 for large |x|)?
   - Sigmoid and Tanh clearly saturate on both ends
   - ReLU saturates only for x < 0
   - ELU, GELU, Swish have soft saturation for x << 0

2. **Which functions are smooth** (no kinks)?
   - Smooth: Sigmoid, Tanh, ELU, GELU, Swish, Softplus
   - Not smooth: Linear (but trivial), ReLU, Leaky ReLU

3. **Which functions can output negative values**?
   - Yes: Linear, Tanh, Leaky ReLU, ELU, GELU, Swish
   - No: Sigmoid, ReLU, Softplus

4. **Which functions have the best gradient flow**?
   - Linear (always 1), ReLU (1 for x>0), Leaky ReLU (never 0)
   - Worst: Sigmoid (max 0.25), Tanh (max 1 but saturates)

### Summary Table

Let's create a comprehensive comparison table:

In [ ]:
import pandas as pd

# Create comparison table
comparison_data = {
    'Function': ['Linear', 'Sigmoid', 'Tanh', 'ReLU', 'Leaky ReLU', 'ELU', 'GELU', 'Swish/SiLU', 'Softplus'],
    'Range': ['(-∞,∞)', '(0,1)', '(-1,1)', '[0,∞)', '(-∞,∞)', '(-α,∞)', '(-0.17,∞)', '(-0.28,∞)', '(0,∞)'],
    'Smooth': ['Yes', 'Yes', 'Yes', 'No', 'No', 'Yes', 'Yes', 'Yes', 'Yes'],
    'Zero-Centered': ['Yes', 'No', 'Yes', 'No', 'No', 'Nearly', 'Nearly', 'Nearly', 'No'],
    'Saturates': ['No', 'Both', 'Both', 'Left', 'No', 'Soft', 'Soft', 'Soft', 'Soft'],
    'Dying Neurons': ['No', 'No', 'No', 'Yes', 'No', 'No', 'No', 'No', 'No'],
    'Compute Cost': ['Very Low', 'Medium', 'Medium', 'Very Low', 'Very Low', 'Medium', 'Med-High', 'Medium', 'Medium'],
    'Common Use': ['Output (regression)', 'Output (binary)', 'RNN/LSTM', 'Hidden (CNN)', 'Hidden', 'Hidden', 'Transformers', 'CNN', 'VAE']
}

df = pd.DataFrame(comparison_data)
print("\n" + "="*100)
print("COMPREHENSIVE ACTIVATION FUNCTION COMPARISON")
print("="*100)
print(df.to_string(index=False))
print("="*100 + "\n")

## Part 7: Practical Demonstration - Training Neural Networks

Let's train simple neural networks with different activations on a real problem and compare their performance!

### Create a Nonlinear Classification Dataset

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# Generate nonlinear dataset (moons)
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to PyTorch tensors
X_train_t = torch.FloatTensor(X_train).to(device)
y_train_t = torch.LongTensor(y_train).to(device)
X_test_t = torch.FloatTensor(X_test).to(device)
y_test_t = torch.LongTensor(y_test).to(device)

# Create dataloaders
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Visualize dataset
plt.figure(figsize=(8, 6))
plt.scatter(X_train[y_train==0, 0], X_train[y_train==0, 1], c='blue', marker='o', 
            s=50, alpha=0.6, label='Class 0', edgecolors='k')
plt.scatter(X_train[y_train==1, 0], X_train[y_train==1, 1], c='red', marker='s', 
            s=50, alpha=0.6, label='Class 1', edgecolors='k')
plt.xlabel('Feature 1', fontsize=12)
plt.ylabel('Feature 2', fontsize=12)
plt.title('Training Dataset: Two Moons (Nonlinear)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Input dimension: {X_train.shape[1]}")
print(f"Number of classes: {len(np.unique(y))}")

### Define Neural Network with Configurable Activation

In [ ]:
class SimpleNN(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=64, output_dim=2, activation='relu'):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        
        # Select activation function
        self.activation_name = activation
        if activation == 'relu':
            self.activation = nn.ReLU()
        elif activation == 'sigmoid':
            self.activation = nn.Sigmoid()
        elif activation == 'tanh':
            self.activation = nn.Tanh()
        elif activation == 'leaky_relu':
            self.activation = nn.LeakyReLU(0.01)
        elif activation == 'elu':
            self.activation = nn.ELU()
        elif activation == 'gelu':
            self.activation = nn.GELU()
        elif activation == 'silu':
            self.activation = nn.SiLU()
        else:
            raise ValueError(f"Unknown activation: {activation}")
    
    def forward(self, x):
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = self.fc3(x)  # No activation on output (logits)
        return x

# Test the network
test_model = SimpleNN(activation='relu').to(device)
test_output = test_model(X_train_t[:5])
print(f"Model output shape: {test_output.shape}")
print(f"Sample output (logits): {test_output[0].detach().cpu().numpy()}")

### Training Function

In [ ]:
def train_model(model, train_loader, epochs=100, lr=0.01):
    """
    Train a model and return training history.
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    train_losses = []
    train_accs = []
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        correct = 0
        total = 0
        
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item() * X_batch.size(0)
            _, predicted = outputs.max(1)
            total += y_batch.size(0)
            correct += predicted.eq(y_batch).sum().item()
        
        epoch_loss /= total
        epoch_acc = 100.0 * correct / total
        train_losses.append(epoch_loss)
        train_accs.append(epoch_acc)
        
        if (epoch + 1) % 20 == 0:
            print(f"  Epoch {epoch+1}/{epochs}: Loss={epoch_loss:.4f}, Acc={epoch_acc:.2f}%")
    
    return train_losses, train_accs

def evaluate_model(model, X_test, y_test):
    """
    Evaluate model on test set.
    """
    model.eval()
    with torch.no_grad():
        outputs = model(X_test)
        _, predicted = outputs.max(1)
        accuracy = predicted.eq(y_test).float().mean().item() * 100
    return accuracy

### Train Models with Different Activations

In [ ]:
# Activations to compare
activations = ['sigmoid', 'tanh', 'relu', 'leaky_relu', 'elu', 'gelu', 'silu']

results = {}
set_seed(42)  # For reproducible comparison

print("Training neural networks with different activation functions...\n")
print("="*70)

for act in activations:
    print(f"\nTraining with {act.upper()}...")
    
    # Create model
    model = SimpleNN(activation=act, hidden_dim=64).to(device)
    
    # Train
    train_losses, train_accs = train_model(model, train_loader, epochs=100, lr=0.01)
    
    # Evaluate
    test_acc = evaluate_model(model, X_test_t, y_test_t)
    
    results[act] = {
        'model': model,
        'train_losses': train_losses,
        'train_accs': train_accs,
        'test_acc': test_acc
    }
    
    print(f"  Final test accuracy: {test_acc:.2f}%")

print("\n" + "="*70)
print("Training complete!\n")

### Compare Training Curves

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Loss curves
for act in activations:
    axes[0].plot(results[act]['train_losses'], label=act.upper(), linewidth=2, alpha=0.8)

axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Training Loss', fontsize=12)
axes[0].set_title('Training Loss by Activation Function', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Accuracy curves
for act in activations:
    axes[1].plot(results[act]['train_accs'], label=act.upper(), linewidth=2, alpha=0.8)

axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Training Accuracy (%)', fontsize=12)
axes[1].set_title('Training Accuracy by Activation Function', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final test accuracies
print("\nFinal Test Accuracies:")
print("="*40)
for act in activations:
    test_acc = results[act]['test_acc']
    print(f"{act.upper():15s}: {test_acc:6.2f}%")
print("="*40)

### 🤔 Observations

From the training curves, you should notice:

1. **Sigmoid and Tanh** typically converge slower due to vanishing gradients
2. **ReLU and variants** (Leaky ReLU, ELU) converge faster
3. **Modern activations** (GELU, SiLU) often achieve competitive or better final accuracy
4. **Convergence speed** varies - some functions reach high accuracy in fewer epochs

This demonstrates why ReLU became the default and why modern architectures use GELU/SiLU!

### Visualize Decision Boundaries

Let's see how different activations learn different decision boundaries:

In [ ]:
def plot_decision_boundary(model, X, y, title):
    """
    Plot the decision boundary learned by a model.
    """
    h = 0.02  # Step size in mesh
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Predict on mesh
    mesh_input = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()]).to(device)
    model.eval()
    with torch.no_grad():
        Z = model(mesh_input)
        Z = F.softmax(Z, dim=1)[:, 1].cpu().numpy()
    Z = Z.reshape(xx.shape)
    
    # Plot
    plt.contourf(xx, yy, Z, levels=20, cmap='RdBu', alpha=0.6)
    plt.colorbar(label='P(Class 1)', fraction=0.046, pad=0.04)
    plt.scatter(X[y==0, 0], X[y==0, 1], c='blue', marker='o', s=30, 
                edgecolors='k', linewidth=0.5, label='Class 0', alpha=0.8)
    plt.scatter(X[y==1, 0], X[y==1, 1], c='red', marker='s', s=30, 
                edgecolors='k', linewidth=0.5, label='Class 1', alpha=0.8)
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.title(title, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.2)

# Plot decision boundaries for selected activations
selected_acts = ['sigmoid', 'relu', 'gelu', 'silu']
fig = plt.figure(figsize=(16, 12))

for idx, act in enumerate(selected_acts, 1):
    plt.subplot(2, 2, idx)
    model = results[act]['model']
    test_acc = results[act]['test_acc']
    plot_decision_boundary(model, X_test, y_test, 
                          f'{act.upper()} (Test Acc: {test_acc:.2f}%)')

plt.tight_layout()
plt.show()

print("\n💡 Notice how different activations learn slightly different decision boundaries!")
print("   Modern activations (GELU, SiLU) often produce smoother, more confident boundaries.")

## Part 8: Advanced Topics

Let's explore some advanced concepts related to activation functions.

### Vanishing and Exploding Gradients

One of the most important considerations when choosing activation functions is how they affect gradient flow during backpropagation.

In [ ]:
def simulate_gradient_flow(activation_fn, depth=10, input_val=0.5):
    """
    Simulate gradient flow through multiple layers.
    
    Args:
        activation_fn: Activation function (callable)
        depth: Number of layers
        input_val: Initial input value
    """
    x = torch.tensor([input_val], requires_grad=True)
    
    # Forward pass through multiple layers
    activations = [x]
    for _ in range(depth):
        x = activation_fn(x)
        activations.append(x)
    
    # Backward pass
    x.backward()
    
    return activations[0].grad.item()

# Test different activations
depths = list(range(1, 21))
gradients = {
    'Sigmoid': [],
    'Tanh': [],
    'ReLU': [],
    'Leaky ReLU': [],
    'ELU': []
}

for depth in depths:
    gradients['Sigmoid'].append(simulate_gradient_flow(torch.sigmoid, depth, 0.5))
    gradients['Tanh'].append(simulate_gradient_flow(torch.tanh, depth, 0.5))
    gradients['ReLU'].append(simulate_gradient_flow(F.relu, depth, 0.5))
    gradients['Leaky ReLU'].append(simulate_gradient_flow(lambda x: F.leaky_relu(x, 0.01), depth, 0.5))
    gradients['ELU'].append(simulate_gradient_flow(F.elu, depth, 0.5))

# Plot gradient magnitude vs depth
plt.figure(figsize=(14, 6))

for name, grads in gradients.items():
    plt.semilogy(depths, np.abs(grads), linewidth=2.5, marker='o', markersize=4, label=name, alpha=0.8)

plt.axhline(y=1.0, color='gray', linestyle='--', linewidth=1.5, alpha=0.5, label='Gradient = 1 (ideal)')
plt.xlabel('Network Depth (Number of Layers)', fontsize=12)
plt.ylabel('Gradient Magnitude (log scale)', fontsize=12)
plt.title('Gradient Flow Through Deep Networks', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print("\n💡 Key Observations:")
print("   - Sigmoid gradients VANISH exponentially with depth (gradient → 0)")
print("   - Tanh is better but still vanishes")
print("   - ReLU maintains gradient magnitude (gradient = 1 for positive inputs)")
print("   - Leaky ReLU and ELU also maintain gradients well")
print("\n   This is why ReLU revolutionized deep learning - it enables training deep networks!")

### Activation Statistics

Let's examine how different activations affect the distribution of neuron outputs:

In [ ]:
# Generate random inputs
torch.manual_seed(42)
random_inputs = torch.randn(10000)

# Apply different activations
activation_outputs = {
    'Input (N(0,1))': random_inputs.numpy(),
    'Sigmoid': torch.sigmoid(random_inputs).numpy(),
    'Tanh': torch.tanh(random_inputs).numpy(),
    'ReLU': F.relu(random_inputs).numpy(),
    'Leaky ReLU': F.leaky_relu(random_inputs, 0.01).numpy(),
    'ELU': F.elu(random_inputs).numpy(),
    'GELU': F.gelu(random_inputs).numpy()
}

# Plot distributions
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for idx, (name, outputs) in enumerate(activation_outputs.items()):
    if idx < len(axes):
        axes[idx].hist(outputs, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
        axes[idx].axvline(x=outputs.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean={outputs.mean():.3f}')
        axes[idx].axvline(x=0, color='gray', linestyle='-', linewidth=1, alpha=0.5)
        axes[idx].set_xlabel('Activation Value')
        axes[idx].set_ylabel('Frequency')
        axes[idx].set_title(f'{name}\n(Mean={outputs.mean():.3f}, Std={outputs.std():.3f})', fontweight='bold')
        axes[idx].legend()
        axes[idx].grid(True, alpha=0.3)

# Remove extra subplots
for idx in range(len(activation_outputs), len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.show()

print("\n💡 Observations:")
print("   - Sigmoid outputs are NOT zero-centered (mean ≈ 0.5)")
print("   - Tanh outputs ARE zero-centered (mean ≈ 0)")
print("   - ReLU outputs are all positive (mean > 0, many zeros)")
print("   - GELU, ELU maintain near-zero mean with some negative values")
print("\n   Zero-centered activations generally train faster!")

## Part 9: Decision Guide - Which Activation to Use?

Here's a practical guide for choosing activation functions based on your use case.

### Quick Decision Tree

```
START: What kind of layer is this?
│
├─ OUTPUT LAYER
│  ├─ Binary Classification → Sigmoid
│  ├─ Multi-class Classification → Softmax
│  ├─ Regression (continuous) → Linear (no activation)
│  └─ Regression (positive only) → ReLU or Softplus
│
└─ HIDDEN LAYER
   ├─ General Purpose (default) → ReLU
   │
   ├─ Experiencing dying ReLU? → Leaky ReLU or ELU
   │
   ├─ Deep network (>10 layers)?
   │  ├─ Yes → ReLU, Leaky ReLU, or ELU (avoid Sigmoid/Tanh)
   │  └─ No → ReLU or Tanh
   │
   ├─ Transformer/NLP model? → GELU
   │
   ├─ Modern CNN? → ReLU, Swish/SiLU, or GELU
   │
   ├─ RNN/LSTM? → Tanh (standard) or ReLU
   │
   └─ Want smooth gradients? → ELU, GELU, or Swish
```

### Recommended Defaults by Architecture

| Architecture | Hidden Layers | Output Layer | Notes |
|--------------|---------------|--------------|-------|
| **Feedforward NN** | ReLU or Leaky ReLU | Task-dependent | ReLU is the safe default |
| **CNN** | ReLU or Swish | Task-dependent | Swish used in EfficientNet |
| **ResNet** | ReLU | Task-dependent | Standard choice |
| **Transformer** | GELU | Softmax (attention) | GELU is the modern standard |
| **RNN/LSTM** | Tanh | Task-dependent | Tanh is traditional for RNNs |
| **GAN** | Leaky ReLU | Tanh (generator) | Prevent mode collapse |
| **VAE** | ReLU or ELU | Sigmoid/Softplus | Softplus for positive variance |
| **Autoencoder** | ReLU | Linear or Sigmoid | Depends on input range |

### Summary of Recommendations

**🏆 Top Choices:**
1. **ReLU**: Default for most cases, fast and effective
2. **GELU**: Modern standard for Transformers and NLP
3. **Leaky ReLU**: When ReLU causes dying neurons
4. **Tanh**: For RNNs and when zero-centered outputs matter
5. **Softmax**: Required for multi-class classification output

**⚠️ Use With Caution:**
- **Sigmoid**: Only for binary classification output or gates (LSTM)
- **Linear**: Only for regression output, never hidden layers
- **Softplus**: Niche uses (VAE, when you need smooth positive outputs)

**❌ Generally Avoid:**
- Sigmoid in hidden layers (vanishing gradients)
- Tanh in very deep networks (vanishing gradients)
- Linear in hidden layers (defeats purpose of depth)

## Part 10: Key Takeaways and Further Exploration

### 🎯 Core Concepts You Should Now Understand

1. **Why we need activation functions**: Without nonlinearity, deep networks collapse to linear models

2. **The vanishing gradient problem**: Sigmoid/Tanh saturate and kill gradients in deep networks

3. **ReLU's revolution**: Simple, fast, and enables deep learning by preserving gradients

4. **Dying ReLU problem**: ReLU neurons can "die" - fixed by Leaky ReLU, ELU

5. **Modern activations**: GELU, Swish are smooth, work well in state-of-the-art models

6. **Zero-centering matters**: Tanh > Sigmoid for hidden layers because of zero-centered outputs

7. **Output layer activations**: Sigmoid (binary), Softmax (multi-class), Linear (regression)

8. **Trade-offs**: Speed vs smoothness, simplicity vs performance

### 🔬 Experiments You Can Try

1. **Change the dataset**: Try different classification problems (MNIST, CIFAR-10)
2. **Vary network depth**: How does depth affect which activation works best?
3. **Mix activations**: Use different activations in different layers
4. **Tune hyperparameters**: What happens with different learning rates?
5. **Test dying ReLU**: Intentionally create conditions that kill ReLU neurons
6. **Implement custom activations**: Create your own activation function!

### 📚 Further Reading

- **Original Papers**:
  - ReLU: "Rectified Linear Units Improve Restricted Boltzmann Machines" (Nair & Hinton, 2010)
  - GELU: "Gaussian Error Linear Units" (Hendrycks & Gimpel, 2016)
  - Swish: "Searching for Activation Functions" (Ramachandran et al., 2017)
  - ELU: "Fast and Accurate Deep Network Learning by Exponential Linear Units" (Clevert et al., 2015)

- **Deep Learning Books**:
  - Deep Learning (Goodfellow, Bengio, Courville) - Chapter 6
  - Neural Networks and Deep Learning (Nielsen) - Chapter 3

### 🎓 You're Ready!

You now have strong intuitions about:
- ✅ What each activation function does mathematically
- ✅ How they behave during training (gradient flow)
- ✅ Their pros, cons, and use cases
- ✅ How to choose the right activation for your problem

**Go build amazing neural networks!** 🚀

---

## 🤔 Final Reflection Questions

Test your understanding:

1. **Why does stacking linear layers without activation result in a linear model?**
   <details>
   <summary>Click for answer</summary>
   Matrix multiplication is associative: (W3 @ W2 @ W1) @ x = W_combined @ x. Multiple linear transformations compose into a single linear transformation.
   </details>

2. **Why is ReLU preferred over Sigmoid for hidden layers in deep networks?**
   <details>
   <summary>Click for answer</summary>
   ReLU has gradient 1 for positive inputs (no vanishing), is faster to compute, and creates sparse activations. Sigmoid saturates (gradient → 0) for large |x|, causing vanishing gradients in deep networks.
   </details>

3. **When would you choose Leaky ReLU over regular ReLU?**
   <details>
   <summary>Click for answer</summary>
   When you observe dying ReLU (neurons always outputting 0), or as a safer default. The small negative slope (0.01) prevents neurons from dying completely.
   </details>

4. **Why is Tanh better than Sigmoid for hidden layers?**
   <details>
   <summary>Click for answer</summary>
   Tanh is zero-centered (outputs in -1 to 1), which leads to faster convergence. It also has 4x larger maximum gradient (1.0 vs 0.25). Both still suffer from saturation though.
   </details>

5. **What activation would you use for the output layer of a 10-class image classifier?**
   <details>
   <summary>Click for answer</summary>
   Softmax - it converts logits into a probability distribution over the 10 classes, with outputs summing to 1.
   </details>